# LLM Grading Consistency Analysis
## Measuring the Statistical Boundary for Calibrated LLM-as-Judge

**Prepared for:** Prof. Will Fithian, UC Berkeley Statistics

---

### Research Question

When using LLMs to evaluate or label data, at what granularity can we trust the model's judgments? We know binary labels (yes/no) are generally accurate, but at what point does multi-class or continuous scoring degrade into noise?

### Methodology

- **Model:** GPT-4o-mini (`temperature=0`)
- **Texts:** 10 diverse statements (factual, sentiment, ambiguous, clinical, etc.)
- **Questions:** 20 mechanistic semantic probes (1 per orthogonal factor family)
- **Scales:** Binary {0,1}, Ternary {0, 0.5, 1}, Quaternary {0, 0.33, 0.66, 1}, Continuous [0,1]
- **Samples:** 20 repeated evaluations per (text, question, scale) triple
- **Total evaluations:** 10 texts × 20 questions × 20 samples × 4 scales = **16,000**

Each evaluation asks the LLM to score how deeply a semantic feature appears in a statement. By repeating 20 times at temperature=0, we measure the model's internal consistency.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_theme(style='whitegrid')

SCALE_ORDER = ['binary', 'ternary', 'quaternary', 'continuous']
SCALE_COLORS = {'binary': '#2196F3', 'ternary': '#4CAF50', 'quaternary': '#FF9800', 'continuous': '#E91E63'}
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

# Load batch results
with open('../data/batch_results.json') as f:
    allResults = json.load(f)

print(f'Loaded {len(allResults)} text results')
print(f'Text IDs: {[r["id"] for r in allResults]}')

In [ ]:
# Load question metadata for labeling
with open('../data/questions_mech.json') as f:
    questionData = json.load(f)
questions = questionData['questions']
questionFamilies = [q['family'] for q in questions]
questionLabels = [q['family'].replace('_', ' ').title() for q in questions]

print(f'Questions: {len(questions)}')
print(f'Factor families: {questionFamilies}')

In [ ]:
# ======================================================================
# Build the master data structure
# varianceMatrix[scale][text_idx][question_idx] = variance of 20 samples
# entropyMatrix[scale][text_idx][question_idx] = entropy of 20 samples
# consistencyMatrix[scale][text_idx][question_idx] = mode consistency
# ======================================================================

nTexts = len(allResults)
nQuestions = 20
nScales = 4

varianceMatrix = {s: np.zeros((nTexts, nQuestions)) for s in SCALE_ORDER}
entropyMatrix = {s: np.zeros((nTexts, nQuestions)) for s in SCALE_ORDER}
consistencyMatrix = {s: np.zeros((nTexts, nQuestions)) for s in SCALE_ORDER}
meanMatrix = {s: np.zeros((nTexts, nQuestions)) for s in SCALE_ORDER}

# Discrete scale values for entropy computation
scaleValues = {
    'binary': [0, 1],
    'ternary': [0, 0.5, 1],
    'quaternary': [0, 0.33, 0.66, 1],
}

def computeEntropy(values, bins=None):
    """Compute Shannon entropy."""
    if bins is not None:
        counts = np.zeros(len(bins))
        for v in values:
            idx = np.argmin(np.abs(np.array(bins) - v))
            counts[idx] += 1
    else:
        counts, _ = np.histogram(values, bins=10)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def computeModeConsistency(values):
    """Fraction of values matching the mode."""
    rounded = np.round(values, 2)
    unique, counts = np.unique(rounded, return_counts=True)
    return counts.max() / len(values)

for tIdx, result in enumerate(allResults):
    for scale in SCALE_ORDER:
        rawScores = np.array(result['raw_scores'][scale])  # (20, 20)
        for qIdx in range(nQuestions):
            samples = rawScores[:, qIdx]  # 20 repeated scores
            varianceMatrix[scale][tIdx, qIdx] = np.var(samples)
            meanMatrix[scale][tIdx, qIdx] = np.mean(samples)
            consistencyMatrix[scale][tIdx, qIdx] = computeModeConsistency(samples)
            bins = scaleValues.get(scale)
            entropyMatrix[scale][tIdx, qIdx] = computeEntropy(samples, bins=bins)

print('Data structure built.')
print(f'Shape per scale: {varianceMatrix["binary"].shape} (texts x questions)')
print(f'Total data points per scale: {nTexts * nQuestions} = {nTexts}×{nQuestions}')

---
## 4.1 — Degradation Table

The central result: how does grading quality degrade as scale granularity increases?

In [ ]:
# Build degradation table
rows = []
for scale in SCALE_ORDER:
    allVars = varianceMatrix[scale].flatten()  # 200 values
    allEntropy = entropyMatrix[scale].flatten()
    allConsistency = consistencyMatrix[scale].flatten()
    
    rows.append({
        'Scale': scale.capitalize(),
        'Mean Variance': f'{np.mean(allVars):.4f}',
        'Median Variance': f'{np.median(allVars):.4f}',
        '% Zero-Variance': f'{(allVars == 0).sum() / len(allVars) * 100:.1f}%',
        '% High-Variance (>0.1)': f'{(allVars > 0.1).sum() / len(allVars) * 100:.1f}%',
        'Median Entropy': f'{np.median(allEntropy):.3f}',
        'Mean Mode Consistency': f'{np.mean(allConsistency):.2%}',
    })

degradationDf = pd.DataFrame(rows)
degradationDf = degradationDf.set_index('Scale')
display(degradationDf)

# Save to CSV
degradationDf.to_csv(FIGURES_DIR / 'degradation_table.csv')
print(f'\nSaved to {FIGURES_DIR / "degradation_table.csv"}')

---
## 4.2 — Question Stability Heatmap

Which questions are inherently unstable, and at which granularity? Each cell shows the mean variance for that question-scale pair, averaged across all 10 texts.

In [ ]:
# Build heatmap: (20 questions x 4 scales), each cell = mean variance across 10 texts
heatmapData = np.zeros((nQuestions, nScales))
for sIdx, scale in enumerate(SCALE_ORDER):
    heatmapData[:, sIdx] = varianceMatrix[scale].mean(axis=0)  # mean across texts

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    heatmapData,
    xticklabels=['Binary', 'Ternary', 'Quaternary', 'Continuous'],
    yticklabels=questionLabels,
    cmap='RdYlGn_r',
    annot=True, fmt='.4f',
    cbar_kws={'label': 'Mean Variance (across 10 texts)'},
    linewidths=0.5,
    ax=ax
)
ax.set_title('Question Stability Across Grading Scales', fontweight='bold')
ax.set_xlabel('Grading Scale')
ax.set_ylabel('Question (Factor Family)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'question_stability_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "question_stability_heatmap.png"}')

---
## 4.3 — Wilcoxon Signed-Rank Tests

Testing whether adjacent scales have statistically different variance distributions. These are **paired tests** because each variance value corresponds to the same (text, question) pair.

**H₀:** Variance at scale A = Variance at scale B  
**H₁:** Variance at scale B > Variance at scale A (one-sided)

In [ ]:
print('Wilcoxon Signed-Rank Tests (one-sided: H1 = scale B more variable than A)')
print('=' * 75)

wilcoxonResults = []
for i in range(len(SCALE_ORDER) - 1):
    scaleA = SCALE_ORDER[i]
    scaleB = SCALE_ORDER[i + 1]
    
    varsA = varianceMatrix[scaleA].flatten()  # 200 paired values
    varsB = varianceMatrix[scaleB].flatten()
    
    # Compute differences
    diffs = varsB - varsA
    nonZeroDiffs = diffs[diffs != 0]
    
    if len(nonZeroDiffs) == 0:
        print(f'{scaleA.capitalize():12s} → {scaleB.capitalize():12s}: All differences are zero (identical distributions)')
        wilcoxonResults.append({'Comparison': f'{scaleA}→{scaleB}', 'W': 'N/A', 'p-value': 'N/A', 'n_nonzero': 0})
        continue
    
    stat, pValue = wilcoxon(varsA, varsB, alternative='less')
    
    # Effect size: rank-biserial correlation
    nPairs = len(nonZeroDiffs)
    rankBiserial = 1 - (2 * stat) / (nPairs * (nPairs + 1) / 2) if nPairs > 0 else 0
    
    sig = '***' if pValue < 0.001 else '**' if pValue < 0.01 else '*' if pValue < 0.05 else 'ns'
    
    print(f'{scaleA.capitalize():12s} → {scaleB.capitalize():12s}: W={stat:10.1f}, p={pValue:.6f} {sig}, r_rb={rankBiserial:.3f}, n_nonzero={nPairs}')
    wilcoxonResults.append({
        'Comparison': f'{scaleA}→{scaleB}',
        'W': f'{stat:.1f}',
        'p-value': f'{pValue:.6f}',
        'Significant (α=0.05)': 'Yes' if pValue < 0.05 else 'No',
        'Rank-biserial r': f'{rankBiserial:.3f}',
        'n_nonzero_pairs': nPairs
    })

# Also test binary vs continuous (full jump)
print('\nFull range test:')
varsA = varianceMatrix['binary'].flatten()
varsB = varianceMatrix['continuous'].flatten()
diffs = varsB - varsA
nonZeroDiffs = diffs[diffs != 0]
if len(nonZeroDiffs) > 0:
    stat, pValue = wilcoxon(varsA, varsB, alternative='less')
    sig = '***' if pValue < 0.001 else '**' if pValue < 0.01 else '*' if pValue < 0.05 else 'ns'
    print(f'{"Binary":12s} → {"Continuous":12s}: W={stat:10.1f}, p={pValue:.6f} {sig}')

print('\n' + pd.DataFrame(wilcoxonResults).to_string(index=False))

---
## 4.4 — Variance Distribution Violin Plot

Distribution of variances across all 200 (question × text) pairs for each scale.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

violinData = [varianceMatrix[s].flatten() for s in SCALE_ORDER]
parts = ax.violinplot(violinData, showmeans=True, showmedians=True)

# Color the violins
for i, body in enumerate(parts['bodies']):
    body.set_facecolor(list(SCALE_COLORS.values())[i])
    body.set_alpha(0.7)

ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['Binary', 'Ternary', 'Quaternary', 'Continuous'])
ax.set_ylabel('Variance (across 20 repeated samples)')
ax.set_title('Distribution of LLM Grading Variance by Scale Granularity', fontweight='bold')

# Add mean annotations
for i, scale in enumerate(SCALE_ORDER):
    meanVal = np.mean(violinData[i])
    ax.annotate(f'μ={meanVal:.4f}', xy=(i+1, meanVal), xytext=(i+1.3, meanVal),
                fontsize=9, ha='left', color=list(SCALE_COLORS.values())[i], fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'variance_violins.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "variance_violins.png"}')

---
## 4.5 — Per-Text Degradation Lines

Does the degradation pattern hold for ALL text types or only some? Each line represents one text.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for tIdx, result in enumerate(allResults):
    textId = result['id']
    meanVarPerScale = []
    for scale in SCALE_ORDER:
        meanVarPerScale.append(varianceMatrix[scale][tIdx, :].mean())
    ax.plot([0, 1, 2, 3], meanVarPerScale, marker='o', label=textId, alpha=0.8, linewidth=2)

ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Binary', 'Ternary', 'Quaternary', 'Continuous'])
ax.set_ylabel('Mean Variance (across 20 questions)')
ax.set_xlabel('Grading Scale')
ax.set_title('Grading Variance Degradation by Text Type', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'per_text_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "per_text_degradation.png"}')

---
## 4.6 — Entropy vs Variance Scatter

Are entropy and variance correlated? Does the relationship differ by scale? Each point is one (text, question) pair.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for scale in SCALE_ORDER:
    allVars = varianceMatrix[scale].flatten()
    allEntropy = entropyMatrix[scale].flatten()
    ax.scatter(allVars, allEntropy, alpha=0.35, label=scale.capitalize(),
               color=SCALE_COLORS[scale], s=25, edgecolors='white', linewidths=0.3)

ax.set_xlabel('Variance')
ax.set_ylabel('Shannon Entropy (bits)')
ax.set_title('Entropy vs Variance by Scale (800 data points)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'entropy_vs_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "entropy_vs_variance.png"}')

---
## 4.7 — Mode Consistency Bar Chart

How often does the LLM give the same most-common answer? Grouped by scale, with one bar per text.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

barWidth = 0.08
textIds = [r['id'] for r in allResults]

for sIdx, scale in enumerate(SCALE_ORDER):
    positions = np.arange(nTexts) + sIdx * barWidth
    meanConsistencies = [consistencyMatrix[scale][tIdx, :].mean() for tIdx in range(nTexts)]
    ax.bar(positions, meanConsistencies, barWidth, label=scale.capitalize(),
           color=SCALE_COLORS[scale], alpha=0.85, edgecolor='white', linewidth=0.5)

ax.set_xticks(np.arange(nTexts) + barWidth * 1.5)
ax.set_xticklabels(textIds, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Mode Consistency (across 20 questions)')
ax.set_title('LLM Response Consistency by Text and Scale', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3, label='Perfect consistency')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mode_consistency_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "mode_consistency_bars.png"}')

---
## 4.8 — Text Difficulty Ranking

Which texts are hardest for the LLM to grade consistently? Ranked by overall mean variance across all scales and questions.

In [ ]:
# Compute overall mean variance per text
textDifficulty = []
for tIdx, result in enumerate(allResults):
    overallMeanVar = np.mean([varianceMatrix[s][tIdx, :].mean() for s in SCALE_ORDER])
    textDifficulty.append((result['id'], overallMeanVar, result['text'][:80]))

textDifficulty.sort(key=lambda x: x[1], reverse=True)

print('Text Difficulty Ranking (most unstable first):')
print('=' * 80)
for rank, (textId, meanVar, preview) in enumerate(textDifficulty, 1):
    print(f'  {rank}. {textId:25s} mean_variance={meanVar:.6f}')
    print(f'     "{preview}..."')
    print()

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
ids = [t[0] for t in textDifficulty]
vars_ = [t[1] for t in textDifficulty]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(ids)))
ax.barh(range(len(ids)), vars_, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(ids)))
ax.set_yticklabels(ids, fontsize=9)
ax.set_xlabel('Mean Variance (across all scales and questions)')
ax.set_title('Text Difficulty Ranking — Most Unstable for LLM Grading', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'text_difficulty_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "text_difficulty_ranking.png"}')

---
## 4.9 — Summary Statistics

Key findings for the Fithian meeting.

In [ ]:
# Compute summary values
binaryMeanVar = np.mean(varianceMatrix['binary'])
continuousMeanVar = np.mean(varianceMatrix['continuous'])
ratio = continuousMeanVar / binaryMeanVar if binaryMeanVar > 0 else float('inf')

# Most/least stable question
questionMeanVars = np.zeros(nQuestions)
for scale in SCALE_ORDER:
    questionMeanVars += varianceMatrix[scale].mean(axis=0)
questionMeanVars /= nScales

mostStableQ = questionFamilies[np.argmin(questionMeanVars)]
leastStableQ = questionFamilies[np.argmax(questionMeanVars)]
mostStableQVar = questionMeanVars.min()
leastStableQVar = questionMeanVars.max()

# Most/least stable text
textMeanVars = np.zeros(nTexts)
for scale in SCALE_ORDER:
    textMeanVars += varianceMatrix[scale].mean(axis=1)
textMeanVars /= nScales

mostStableT = allResults[np.argmin(textMeanVars)]['id']
leastStableT = allResults[np.argmax(textMeanVars)]['id']

# Wilcoxon results
pairs = [('binary', 'ternary'), ('ternary', 'quaternary'), ('quaternary', 'continuous')]
wilcoxonSummary = []
for scaleA, scaleB in pairs:
    varsA = varianceMatrix[scaleA].flatten()
    varsB = varianceMatrix[scaleB].flatten()
    diffs = varsB - varsA
    nonZero = diffs[diffs != 0]
    if len(nonZero) > 0:
        stat, pVal = wilcoxon(varsA, varsB, alternative='less')
        sig = 'significant' if pVal < 0.05 else 'not significant'
        wilcoxonSummary.append(f'  {scaleA.capitalize():12s} → {scaleB.capitalize():12s}: p = {pVal:.6f} [{sig} at α=0.05]')
    else:
        wilcoxonSummary.append(f'  {scaleA.capitalize():12s} → {scaleB.capitalize():12s}: identical distributions')

# Total cost
totalCost = sum(r.get('usage', {}).get('cost_usd', 0) for r in allResults)
totalTokens = sum(r.get('usage', {}).get('total_tokens', 0) for r in allResults)

trend = 'increases' if continuousMeanVar > binaryMeanVar else 'stays flat or decreases'

summary = f"""
SUMMARY FOR FITHIAN MEETING
{'=' * 50}
Texts analyzed: {nTexts}
Questions per text: {nQuestions} (mechanistic mode)
Samples per question per scale: 20
Total evaluations: {nTexts * nQuestions * 20 * nScales:,}
Model: gpt-4o-mini (temperature=0)
Total API cost: ${totalCost:.4f}
Total tokens: {totalTokens:,}

Key Finding: Variance {trend} monotonically from binary → continuous.
Binary mean variance:     {binaryMeanVar:.6f}
Continuous mean variance:  {continuousMeanVar:.6f}
Ratio (continuous/binary): {ratio:.2f}x

Wilcoxon tests (one-sided, H1: scale B more variable than scale A):
{chr(10).join(wilcoxonSummary)}

Most stable question:  {mostStableQ} (mean var = {mostStableQVar:.6f})
Least stable question: {leastStableQ} (mean var = {leastStableQVar:.6f})
Most stable text type:  {mostStableT} (mean var = {textMeanVars.min():.6f})
Least stable text type: {leastStableT} (mean var = {textMeanVars.max():.6f})
"""

print(summary)